# 04) Final Model Selection

In [1]:
import json

import numpy as np
import pandas as pd

train = pd.read_csv("data/train.csv", parse_dates=["date"])
with open("data/columns.json") as f:
    columns = json.load(f)

TARGET_COL = columns["target_cols"][0]
FINAL_FEATURE_COLS = columns["final_feature_cols"]

print(f"train: {train.shape[0]} rows, target = {TARGET_COL!r}")
print(f"{len(FINAL_FEATURE_COLS)} final features (03)'s conclusion):", FINAL_FEATURE_COLS)

train: 8443 rows, target = 'target_bps_vs_mid'
4 final features (03)'s conclusion): ['near_price_vs_mid_bps', 'urgency', 'ref_price_vs_mid_bps', 'far_price_vs_mid_bps']


## Train / eval split

`data/test.csv` stays untouched -- still reserved for final validation once a model is actually
chosen. Same day-based 80/20 split as `02)`'s original train/test split, applied here to
`data/train.csv` itself: cumulative row count snapped to a full trading day at the ~80% mark, so no
day's rows land on both sides.

In [2]:
daily_counts = train.groupby("date").size().sort_index()
cum_frac = daily_counts.cumsum() / daily_counts.sum()
split_date = cum_frac[cum_frac >= 0.80].index[0]

fit = train[train["date"] <= split_date].copy()
eval_ = train[train["date"] > split_date].copy()

print(f"split date: {split_date.date()} (last day included in fit)")
print(f"fit:  {fit.shape[0]} rows ({fit.shape[0]/len(train):.1%}), "
      f"{fit['date'].nunique()} days, {fit['date'].min().date()} to {fit['date'].max().date()}")
print(f"eval: {eval_.shape[0]} rows ({eval_.shape[0]/len(train):.1%}), "
      f"{eval_['date'].nunique()} days, {eval_['date'].min().date()} to {eval_['date'].max().date()}")
print()
print("no date overlap:", set(fit["date"]).isdisjoint(set(eval_["date"])))

split date: 2025-03-20 (last day included in fit)
fit:  6952 rows (82.3%), 14 days, 2025-03-03 to 2025-03-20
eval: 1491 rows (17.7%), 3 days, 2025-03-21 to 2025-03-25

no date overlap: True


## LazyPredict: broad model scan

A cheap, breadth-first pass -- `LazyRegressor` fits dozens of sklearn-compatible regressors with
their defaults and reports R^2/RMSE on the held-out `eval` split. This is not a tuned or
cross-validated comparison (single split, default hyperparameters throughout); the point is to
narrow down which *families* of model look worth tuning properly, not to pick a final model off
this table directly.

`far_price_vs_mid_bps` is the only final feature with nulls (unavailable when `far_price_available`
is False) -- median-imputed here (median taken from `fit` only) since most of the models
`LazyRegressor` wraps don't accept `NaN` natively.

In [3]:
fit_far_median = fit["far_price_vs_mid_bps"].median()
for df in (fit, eval_):
    df["far_price_vs_mid_bps"] = df["far_price_vs_mid_bps"].fillna(fit_far_median)

X_fit, y_fit = fit[FINAL_FEATURE_COLS], fit[TARGET_COL]
X_eval, y_eval = eval_[FINAL_FEATURE_COLS], eval_[TARGET_COL]

from lazypredict.Supervised import LazyRegressor

reg = LazyRegressor(verbose=0, ignore_warnings=True)
leaderboard, _ = reg.fit(X_fit, X_eval, y_fit, y_eval)

pd.set_option("display.max_rows", None)
print(leaderboard.to_string())

                               Adjusted R-Squared     R-Squared         RMSE  Time Taken
Model                                                                                   
HuberRegressor                           0.920066      0.920280     2.887451    0.019485
MLPRegressor                             0.917857      0.918078     2.927074    1.256804
LassoCV                                  0.916667      0.916891     2.948200    0.027295
LassoLarsCV                              0.916647      0.916871     2.948556    0.006691
LarsCV                                   0.916647      0.916871     2.948556    0.007391
OrthogonalMatchingPursuitCV              0.915844      0.916070     2.962723    0.006473
LinearRegression                         0.915734      0.915961     2.964651    0.004362
TransformedTargetRegressor               0.915734      0.915961     2.964651    0.004802
LassoLarsIC                              0.915734      0.915961     2.964651    0.004556
Lars                 